<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Word times and alignment

**[Word times and alignment](https://learning.nextia-ai.com/courses/speech/m05/word-times-and-alignment/)** · Speech AI: Transcribe, Understand and Speak · Module 5, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** get a time for every word from Whisper, check those times against the voices' own word times, join each word to a speaker turn, and write a transcript that says who said what and when. You also catch a promise that was given to the wrong speaker.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (Whisper tiny, 75 MB, is downloaded once). No account and no key. |
| **You should know** | Turns and the DER, from [Split by channel first](https://learning.nextia-ai.com/courses/speech/m05/split-by-channel-first/) and [Speaker embeddings and clustering](https://learning.nextia-ai.com/courses/speech/m05/speaker-embeddings-and-clustering/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m05/word-times-and-alignment/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M05-L04-word-times-and-alignment/word-times-and-alignment-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, faster-whisper 1.2.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time. faster-whisper brings its own engine (CTranslate2) and the Silero voice detector: the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2 faster-whisper==1.2.1
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile', 'faster-whisper']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 3 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/calls/calls.jsonl": "0ab1c11490f1e0341f11f9cf0fc061abde616a9297fbc830cde001ef60daa01a",
    "data/calls/wideband/C-0011.flac": "b2422106baaa6a2009ee301f302fc87fe9a1b2be25c47f534bde639ca43a3169",
    "data/calls/phone/C-0108.wav": "5cd79dfe3361933b72be1e901e46b799f83d09adfb45550e18dad623136ba2f4",
    "compare.py": "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Join words to turns: a tiny example

Diarization gives **turns**: (start, end, speaker). A recogniser gives **words**, each with a start and an end time. To write "who said what", we give each word to the turn that holds the **middle** of the word. Then words in a row with the same speaker become one line.

> **Predict.** The agent's turn is 0.0-1.0 s and the caller's turn is 1.2-3.2 s. The word "it?" runs from 0.62 to 0.95 s. Who said it?

In [ ]:
def join(words, turns):
    """words: (word, start, end); turns: (start, end, speaker).
    Each word goes to the turn that holds the middle of the word; "?" when no turn holds it."""
    out = []
    for w, s, e in words:
        mid = (s + e) / 2
        who = next((t[2] for t in turns if t[0] <= mid <= t[1]), "?")
        out.append((who, s, w))
    return out

def lines(joined):
    """Words in a row with the same speaker become one line: [start] speaker: words."""
    out = []
    for who, s, w in joined:
        if out and out[-1][0] == who:
            out[-1][2].append(w)
        else:
            out.append([who, s, [w]])
    return [f"[{s:5.1f} s] {who:6} {' '.join(ws)}" for who, s, ws in out]

tiny_words = [("Can", 0.10, 0.30), ("I", 0.30, 0.38), ("have", 0.38, 0.62), ("it?", 0.62, 0.95),
              ("Yes,", 1.30, 1.55), ("it's", 1.55, 1.80), ("742155.", 1.80, 3.10)]
tiny_turns = [(0.0, 1.0, "agent"), (1.2, 3.2, "caller")]
for line in lines(join(tiny_words, tiny_turns)):
    print(line)

> **Check.** The middle of "it?" is 0.785 s, inside the agent's turn. Two lines: the agent from 0.1 s, the caller from 1.3 s.

## 2. Word times from Whisper

Whisper can give a start and an end for each word (`word_timestamps=True`). It finds them from where its attention looks in the sound, so they are estimates. Call C-0011 is an English call; we use the wideband audio.

In [ ]:
from faster_whisper import WhisperModel
from huggingface_hub import snapshot_download
import re

model = WhisperModel(snapshot_download("Systran/faster-whisper-tiny", revision="d90ca5fe260221311c53c58e660288d3deb8d356"), device="cpu", compute_type="int8", cpu_threads=4)
calls = load_calls()
call = calls["C-0011"]
x, sr = read_audio("data/calls/wideband/C-0011.flac", 16000)
segments, _ = model.transcribe(x, language="en", word_timestamps=True, beam_size=1, condition_on_previous_text=False)
words = [(w.word.strip(), w.start, w.end) for s in segments for w in s.words]
print(len(words), "words; the first five:", [(w, round(float(s), 2), round(float(e), 2)) for w, s, e in words[:5]])

## 3. How good are the word times?

Larkfield's calls are synthetic, so we know when each word really started: the voice reported it. We match each true word to a Whisper word with the same spelling that starts within 1 s, and measure the start error.

> **Predict.** Is the median error closer to 0.01 s, 0.05 s or 0.3 s?

In [ ]:
def plain(w):
    return re.sub(r"[^\w']", "", w).lower()

errors = []
truth_words = [(w["w"], w["start"]) for t in call["turns"] if t["word_times"] == "voice" for w in t["words"]]
for w, start in truth_words:
    near = [s for h, s, e in words if plain(h) == plain(w) and abs(s - start) < 1.0]
    if near:
        errors.append(min(abs(s - start) for s in near))
errors = np.array(errors)
print(f"{len(errors)} of {len(truth_words)} words matched; median start error {np.median(errors):.3f} s; "
      f"90th percentile {np.percentile(errors, 90):.3f} s")

> **Check.** 97 of 113 words matched; median start error 0.074 s, 90th percentile 0.140 s. The course measured Whisper base on 20 English calls: median 0.055 s, 90th percentile 0.137 s. Good enough to give a word to a turn; not good enough to cut a single word out of the audio without a margin.

## 4. A transcript with names and times

Now join Whisper's words to the true turns of the call.

In [ ]:
truth = true_turns(call)
for line in lines(join(words, truth))[:6]:
    print(line)

## 5. A promise under the wrong name

Call C-0108 is stereo, but here we mix it into one channel and use the turns that the project's embedding method found on the mix (pasted in the next cell). The agent promises: "I will email you the setup guide with pictures today."

A note on this call: the agent tells the caller to "charge it fully before you set it up", for a hammock. That is a mistake in our own scripts, not a recognition error.

> **Predict.** Which speaker does the promise get?

In [ ]:
# The turns that the project's embedding method found for C-0108 (a stereo call mixed to mono):
emb_turns = [(0.57, 1.7, 'agent'), (1.7, 3.2, 'caller'), (3.2, 4.7, 'agent'), (4.7, 10.43, 'caller'), (10.65, 14.78, 'caller'), (14.97, 17.28, 'caller'), (17.46, 18.43, 'agent'), (18.43, 19.04, 'caller'), (19.26, 20.38, 'caller'), (20.6, 23.1, 'caller')]
p_call = calls["C-0108"]
y, ysr = read_audio("data/calls/phone/C-0108.wav", 16000)          # both channels mixed
segs, _ = model.transcribe(y, language="en", word_timestamps=True, beam_size=1, condition_on_previous_text=False)
p_words = [(w.word.strip(), w.start, w.end) for s in segs for w in s.words]
p_lines = lines(join(p_words, emb_turns))
for line in p_lines:
    print(line)

> **Check.** Read the line with "I will email you". The embedding method gave it to the caller. A call record built from this transcript would say the caller promised to send a guide, and nobody at Larkfield would send it.

## 6. Catch it with a rule

Larkfield's callers do not promise to send things; agents do. So a line where the **caller** says "I will" or "I have" is suspect. The next cell flags such lines, first with the predicted turns, then with the true turns.

In [ ]:
PROMISE_WORDS = ("i will", "i'll", "i have", "we will")      # how Larkfield's agents start a promise

def promises_by_caller(text_lines):
    """Lines where the caller seems to promise something: a sign that the names are wrong."""
    return [l for l in text_lines if " caller " in f" {l.split('] ', 1)[1]} " and
            any(p in l.lower() for p in PROMISE_WORDS)]

print(promises_by_caller(p_lines))
print(promises_by_caller(lines(join(p_words, true_turns(p_call)))))

> **Check.** With the predicted turns: 1 line(s) flagged. With the true turns: 0. The rule does not fix the names; it sends the call to a person, or to the channel method when the call is stereo.

## 7. Change one thing: the start instead of the middle

> **Predict.** If you give each word to the turn that holds its **start**, do more words or fewer words fall in no turn ("?")?

In [ ]:
def join_by_start(words, turns):
    return [(next((t[2] for t in turns if t[0] <= s <= t[1]), "?"), s, w) for w, s, e in words]

by_mid = join(words, truth)
by_start = join_by_start(words, truth)
print("no turn, by the middle:", sum(w[0] == "?" for w in by_mid), " by the start:", sum(w[0] == "?" for w in by_start))

> **Check.** By the middle: 2 words in no turn. By the start: 4. Whisper's starts are often a little early (before the true start), so the start of a word can fall in the gap before its turn. The middle is safer.

## 8. Your turn: how many word times are close?

> **Your turn.** From `errors` (section 3), compute the share of matched words whose start error is 0.1 s or less, rounded to 3 decimals, and put it in `task_share`.

In [ ]:
task_share = ...

In [ ]:
expect('task_share', task_share, 0.66, tolerance=0.002)

## Next

You joined words to speakers, checked the word times and caught a promise under the wrong name. The [lesson page](https://learning.nextia-ai.com/courses/speech/m05/word-times-and-alignment/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Why did they call?](https://learning.nextia-ai.com/courses/speech/m06/why-did-they-call/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m05/word-times-and-alignment/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.